# SQE state optimisation with an adjoint entropy gradient

This notebook follows the existing SQE state Lagrangian workflow. It first checks the adjoint gradient against the previous central-difference calculation, then runs the constrained optimisation.

For $C(x)=\frac{1}{N+1}\sum_{k=0}^{N}[S(\psi_k)-S_{\max}]_+^2$, write $dC=2\operatorname{Re}\sum_k q_k^\dagger d\psi_k$. With $\psi_{k+1}=U_k\psi_k$, the backward costate obeys $p_N=q_N$ and $p_k=q_k+U_k^\dagger p_{k+1}$. Consequently, $\partial C/\partial x_j=2\operatorname{Re}\sum_{k\in j}p_{k+1}^\dagger(\partial U_k/\partial x_j)\psi_k$. The code differentiates the von Neumann or linear entropy analytically and uses the exact derivative of PySTE's Trotter step.

In [ ]:
import sys
from pathlib import Path
import time
import numpy as np

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pulse_vqe_entanglement").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pulse_vqe_entanglement.dataclasses import SimulationConfig
from pulse_vqe_entanglement.entanglement_measures import STATE_MEASURES
from pulse_vqe_entanglement.sqe_state import (
    _make_system_and_observable, _make_driven_sys, _entropy_grad_fd, _cost_entropy,
)
from pulse_vqe_entanglement.sqe_state_adjoint import sqe_optimisation
from pulse_vqe_entanglement.sqe_state_entropy_gradient import entropy_cost_and_grad
from pulse_vqe_entanglement.plotting_state import plot_sqe_optimisation

## Check the gradient and simulator trajectory

This uses the same two-qubit physics and entropy cap as the earlier SQE notebook on a smaller grid. The finite-difference timing includes two simulator propagations per pulse parameter.

In [ ]:
check_config = SimulationConfig(
    zeeman_splittings=np.array([1000.0, 1008.0]),
    initial_state=np.array([0, 1, 0, 0], dtype=np.complex128),
    theta_target=0.01 * np.pi,
    n_time_steps=12,
    n_discretisation=2,
    T=500.0,
    J_max=10.0,
    J_min=0.0,
    feromagnetic=False,
    S_max=0.01,
    state_ent_measure="von_neumann",
    J_init_seed=1234,
)
check_x = np.random.default_rng(1234).uniform(-0.5, 0.5, check_config.n_time_steps)
sys, observable = _make_system_and_observable(check_config)
driven_sys = _make_driven_sys(check_config, sys)
driven_sys.propagate_all(check_x)  # initialise PySTE

start = time.perf_counter()
cost, analytic_grad, model_traj = entropy_cost_and_grad(
    check_config, check_x, return_trajectory=True
)
adjoint_seconds = time.perf_counter() - start

sim_traj = np.asarray(driven_sys.propagate_all(check_x))
_, ent_traj_fn = STATE_MEASURES[check_config.state_ent_measure]
start = time.perf_counter()
fd_grad = _entropy_grad_fd(
    driven_sys, check_x, S_max=check_config.S_max,
    eps=1e-6, ent_traj_fn=ent_traj_fn,
)
fd_seconds = time.perf_counter() - start
sim_cost = _cost_entropy(
    driven_sys, check_x, S_max=check_config.S_max,
    ent_traj_fn=ent_traj_fn,
)
print(f"Trajectory max error: {np.max(np.abs(model_traj - sim_traj)):.3e}")
print(f"Cost error:           {abs(cost - sim_cost):.3e}")
print(f"Gradient max error:   {np.max(np.abs(analytic_grad - fd_grad)):.3e}")
print(f"Adjoint: {adjoint_seconds:.3f} s; finite difference: {fd_seconds:.3f} s; speedup: {fd_seconds / adjoint_seconds:.1f}x")
assert np.allclose(model_traj, sim_traj, atol=1e-7)
assert np.allclose(analytic_grad, fd_grad, atol=1e-5, rtol=1e-4)

## Run the augmented Lagrangian method

The earlier finite-difference SQE notebook starts its entropy stage from the original pulse and updates a nonnegative multiplier after each L-BFGS-B solve. This run uses the same initial pulse, entropy cap, fidelity tolerance, dual learning rate, and outer-loop settings. The new inner solve uses the adjoint entropy gradient and a tighter relative function-change tolerance so it can continue reducing a small cost.

In [ ]:
config = SimulationConfig(
    zeeman_splittings=np.array([1000.0, 1008.0]),
    initial_state=np.array([0, 1, 0, 0], dtype=np.complex128),
    theta_target=0.01 * np.pi,
    n_time_steps=50,
    n_discretisation=1,
    T=500.0,
    max_drive_strength=0.0,
    J_max=10.0,
    J_min=0.0,
    J_init_scale=1.0,
    J_init_mean=0.0,
    J_init_seed=1234,
    feromagnetic=False,
    gtol=1e-12,
    maxiter=200,
    entanglement_optimisation=True,
    state_ent_measure="von_neumann",
    S_max=0.01,
    eps_infidelity=1e-8,
    rho=0.0,
    dual_lr=1e10,
    n_outer=2,
    inner_maxiter=200,
    fd_eps=1e-6,  # used by the optional finite-difference comparison below
)
start = time.perf_counter()
results = sqe_optimisation(config)
adjoint_run_seconds = time.perf_counter() - start
ent_traj_fn = STATE_MEASURES[config.state_ent_measure][1]
adjoint_entropy = ent_traj_fn(results.final_traj)
print(f"Final infidelity: {float(results.val_final[-1]):.3e}")
print(f"Entropy cost:     {results.history[-1]['S']:.3e}")
print(f"Maximum entropy:  {np.max(adjoint_entropy):.6f} bits")
print(f"Fidelity feasible: {results.optimization_success}")
print(f"Elapsed:          {adjoint_run_seconds:.3f} s")
plot_sqe_optimisation(results)

## Compare with the earlier finite-difference optimizer

This reruns the original augmented Lagrangian method with the same configuration. It takes longer but gives a direct comparison of entropy suppression, fidelity, and total run time. For routine adjoint runs, this cell can be skipped.

In [ ]:
from pulse_vqe_entanglement.sqe_state import sqe_optimisation as sqe_fd_optimisation

start = time.perf_counter()
fd_results = sqe_fd_optimisation(config)
fd_run_seconds = time.perf_counter() - start
fd_entropy = ent_traj_fn(fd_results.final_traj)
fd_cost = np.mean(np.maximum(fd_entropy - config.S_max, 0.0) ** 2)

print(f"{'Method':<20} {'Infidelity':>14} {'Entropy cost':>14} {'Max entropy':>14} {'Time (s)':>10}")
print(f"{'Adjoint':<20} {float(results.val_final[-1]):14.3e} {results.history[-1]['S']:14.3e} {np.max(adjoint_entropy):14.6f} {adjoint_run_seconds:10.3f}")
print(f"{'Finite difference':<20} {float(fd_results.val_final[-1]):14.3e} {fd_cost:14.3e} {np.max(fd_entropy):14.6f} {fd_run_seconds:10.3f}")
assert results.optimization_success
assert fd_results.optimization_success